In [3]:
import pandas as pd
import numpy as np
import pymc as pm
import scipy.stats as stats
import arviz as az

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, minmax_scale

from gurobipy import *
import pytensor.tensor as pt

import joblib
from collections import Counter
import collections
import random
from tqdm import tqdm

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


## Utils


In [118]:
scaler_gk = joblib.load('./scaler_gk.pkl')
scaler_def = joblib.load('./scaler_def.pkl')
scaler_mid = joblib.load('./scaler_mid.pkl')
scaler_fwd = joblib.load('./scaler_fwd.pkl')

pca_gk = joblib.load('./pca_gk.pkl')
pca_def = joblib.load('./pca_def.pkl')
pca_mid = joblib.load('./pca_mid.pkl')
pca_fwd = joblib.load('./pca_fwd.pkl')

beta_gk = pd.read_csv('./gk_beta.csv').values.squeeze()
beta_def = pd.read_csv('./def_beta.csv').values.squeeze()
beta_mid = pd.read_csv('./mid_beta.csv').values.squeeze()
beta_fwd = pd.read_csv('./fwd_beta.csv').values.squeeze()

data_cols = joblib.load('./data_cols')

len(data_cols)


107

In [ ]:
data_cols_gk = ['clean_sheets',  'creativity', 'ict_index',  'influence',  'selected',  'threat',  'transfers_balance',
            'transfers_in',  'transfers_out',  'value',   'team_h_difficulty',  'team_a_difficulty', 'ownership_change',  'percenatge_net_transfers',
            'whh',  'whd',  'wha',

            'clean_sheets_1',  'expected_assists_1',  'expected_goal_involvements_1',
            'expected_goals_1',  'expected_goals_conceded_1',  'goals_conceded_1',  'goals_scored_1',  'ict_index_1',  'influence_1',
            'creativity_1',  'threat_1',  'minutes_1',  'own_goals_1',  'penalties_missed_1',  'penalties_saved_1',  'red_cards_1',
            'yellow_cards_1',  'saves_1',  'starts_1',  'team_a_score_1',  'team_h_score_1',  'total_points_1',  'goals_1',  'shots_1',
            'xG_1',  'xA_1',  'assists_x_1',  'key_passes_1',  'npg_1',  'npxG_1',  'xGChain_1',  'xGBuildup_1',  'xP_1',

            'clean_sheets_3',  'expected_assists_3',  'expected_goal_involvements_3',  'expected_goals_3',  'expected_goals_conceded_3',
            'goals_conceded_3',  'goals_scored_3',  'ict_index_3',  'influence_3',  'creativity_3',  'threat_3',  'minutes_3',  'own_goals_3',
            'penalties_missed_3',  'penalties_saved_3',  'red_cards_3',  'yellow_cards_3',  'saves_3',  'starts_3',  'team_a_score_3',
            'team_h_score_3',  'total_points_3',  'goals_3',  'shots_3',  'xG_3',  'xA_3',  'assists_x_3', 'key_passes_3',
            'npg_3',  'npxG_3',  'xGChain_3',  'xGBuildup_3',  'xP_3',

            'clean_sheets_5',  'expected_assists_5',  'expected_goal_involvements_5',  'expected_goals_5',  'expected_goals_conceded_5',  'goals_conceded_5',
            'goals_scored_5',  'ict_index_5',  'influence_5',  'creativity_5',  'threat_5',  'minutes_5',  'own_goals_5',  'penalties_missed_5',  'penalties_saved_5',
            'red_cards_5',  'yellow_cards_5',  'saves_5',  'starts_5',  'team_a_score_5',  'team_h_score_5',  'total_points_5',  'goals_5',  'shots_5',  'xG_5',  'xA_5',
            'assists_x_5', 'key_passes_5',  'npg_5',  'npxG_5',  'xGChain_5',  'xGBuildup_5',  'xP_5'
        ]

data_cols_gk = [
                'clean_sheets', 'creativity', 'ict_index', 'influence', 'selected', 'threat', 'transfers_balance', 'transfers_in', 'transfers_out', 'value',
                'team_h_difficulty', 'team_a_difficulty', 'ownership_change', 'percenatge_net_transfers', 'whh',  'whd',  'wha',

                'clean_sheets_1', 'expected_goals_conceded_1', 'goals_conceded_1', 'ict_index_1', 'influence_1', 'creativity_1', 'threat_1', 'minutes_1', 'own_goals_1',
                'penalties_saved_1', 'red_cards_1', 'yellow_cards_1', 'saves_1', 'starts_1', 'team_a_score_1', 'team_h_score_1', 'total_points_1',  'goals_1', 'xG_1',
                'key_passes_1', 'xGChain_1', 'xGBuildup_1', 'xP_1',

                'clean_sheets_3', 'expected_goals_conceded_3', 'goals_conceded_3', 'ict_index_3', 'influence_3', 'creativity_3', 'threat_3', 'minutes_3', 'own_goals_3',
                'red_cards_3', 'yellow_cards_3', 'saves_3', 'starts_3', 'team_a_score_3', 'team_h_score_3', 'total_points_3', 'goals_3', 'shots_3', 'key_passes_3',
                'xGChain_3', 'xGBuildup_3', 'xP_3',

                'clean_sheets_5', 'expected_goals_conceded_5',  'goals_conceded_5', 'ict_index_5',  'influence_5',  'creativity_5',  'threat_5',  'minutes_5',  'own_goals_5',
                'penalties_saved_5', 'red_cards_5',  'yellow_cards_5',  'saves_5',  'starts_5',  'team_a_score_5',  'team_h_score_5', 'total_points_5',  'goals_5',  'shots_5',
                'key_passes_5', 'xGChain_5',  'xGBuildup_5', 'xP_5'
                ]

data_cols = [
                'clean_sheets', 'creativity', 'ict_index', 'influence', 'selected', 'threat', 'transfers_balance', 'transfers_in', 'transfers_out', 'value',
                'team_h_difficulty',  'team_a_difficulty', 'ownership_change', 'percenatge_net_transfers', 'whh', 'whd', 'wha',

                'clean_sheets_1', 'expected_assists_1', 'expected_goal_involvements_1', 'expected_goals_1', 'expected_goals_conceded_1', 'goals_conceded_1',
                'goals_scored_1', 'ict_index_1', 'influence_1', 'creativity_1', 'threat_1', 'minutes_1', 'red_cards_1', 'yellow_cards_1', 'saves_1', 'starts_1',
                'team_a_score_1', 'team_h_score_1', 'total_points_1', 'goals_1', 'shots_1', 'xG_1', 'xA_1', 'assists_x_1', 'key_passes_1', 'npg_1', 'npxG_1',
                'xGChain_1', 'xGBuildup_1', 'xP_1',

                'clean_sheets_3', 'expected_assists_3', 'expected_goal_involvements_3', 'expected_goals_3', 'expected_goals_conceded_3', 'goals_conceded_3',
                'goals_scored_3', 'ict_index_3', 'influence_3', 'creativity_3', 'threat_3', 'minutes_3', 'red_cards_3', 'yellow_cards_3', 'saves_3', 'starts_3',
                'team_a_score_3', 'team_h_score_3', 'total_points_3', 'goals_3', 'shots_3', 'xG_3', 'xA_3', 'assists_x_3', 'key_passes_3', 'npg_3', 'npxG_3',
                'xGChain_3', 'xGBuildup_3', 'xP_3',

                'clean_sheets_5', 'expected_assists_5', 'expected_goal_involvements_5', 'expected_goals_5', 'expected_goals_conceded_5', 'goals_conceded_5',
                'goals_scored_5', 'ict_index_5', 'influence_5', 'creativity_5', 'threat_5', 'minutes_5', 'red_cards_5', 'yellow_cards_5', 'saves_5', 'starts_5',
                'team_a_score_5', 'team_h_score_5', 'total_points_5', 'goals_5', 'shots_5', 'xG_5', 'xA_5', 'assists_x_5', 'key_passes_5', 'npg_5', 'npxG_5',
                'xGChain_5', 'xGBuildup_5', 'xP_5',
        ]


In [109]:
def clean_data(players_preds):
    gk_data = players_preds[players_preds['position'] == 1][data_cols_gk]
    def_data = players_preds[players_preds['position'] == 2][data_cols]
    mid_data = players_preds[players_preds['position'] == 3][data_cols]
    fwd_data = players_preds[players_preds['position'] == 4][data_cols]
    data = [gk_data, def_data, mid_data, fwd_data]

    # Get attributes
    g_prices = players_preds[players_preds['position'] == 1][['value']] #*10
    d_prices = players_preds[players_preds['position'] == 2][['value']] #*10
    m_prices = players_preds[players_preds['position'] == 3][['value']] #*10
    f_prices = players_preds[players_preds['position'] == 4][['value']] #*10
    prices = [g_prices, d_prices, m_prices, f_prices]

    g_xPts = players_preds[players_preds['position'] == 1]['xP_preds'].tolist()
    d_xPts = players_preds[players_preds['position'] == 2]['xP_preds'].tolist()
    m_xPts = players_preds[players_preds['position'] == 3]['xP_preds'].tolist()
    f_xPts = players_preds[players_preds['position'] == 4]['xP_preds'].tolist()
    xPts = [g_xPts, d_xPts, m_xPts, f_xPts]

    g_teams = players_preds[players_preds['position'] == 1]['player_team'].tolist()
    d_teams = players_preds[players_preds['position'] == 2]['player_team'].tolist()
    m_teams = players_preds[players_preds['position'] == 3]['player_team'].tolist()
    f_teams = players_preds[players_preds['position'] == 4]['player_team'].tolist()
    teams = [g_teams, d_teams, m_teams, f_teams]

    g_ids = players_preds[players_preds['position'] == 1]['FPL_ID'].tolist()
    d_ids = players_preds[players_preds['position'] == 2]['FPL_ID'].tolist()
    m_ids = players_preds[players_preds['position'] == 3]['FPL_ID'].tolist()
    f_ids = players_preds[players_preds['position'] == 4]['FPL_ID'].tolist()
    ids = [g_ids, d_ids, m_ids, f_ids]

    gk_data = gk_data.select_dtypes(include=[np.number])
    gk_data = pd.DataFrame(minmax_scale(gk_data), columns=gk_data.columns, index=gk_data.index)

    def_data = def_data.select_dtypes(include=[np.number])
    # Data
    def_data = pd.DataFrame(minmax_scale(def_data), columns=def_data.columns, index=def_data.index)

    mid_data = mid_data.select_dtypes(include=[np.number])
    mid_data = pd.DataFrame(minmax_scale(mid_data), columns=mid_data.columns, index=mid_data.index)

    fwd_data = fwd_data.select_dtypes(include=[np.number])
    fwd_data = pd.DataFrame(minmax_scale(fwd_data), columns=fwd_data.columns, index=fwd_data.index)

    # Make sure there are no nan or inf
    def clean_array(arr):
        arr = np.nan_to_num(arr, nan=0.0, posinf=1e6, neginf=-1e6)
        return arr

    gk_data  = clean_array(gk_data)
    def_data = clean_array(def_data)
    mid_data = clean_array(mid_data)
    fwd_data = clean_array(fwd_data)

    X_gk = scaler_gk.transform(gk_data)
    X_def = scaler_def.transform(def_data)
    X_mid = scaler_mid.transform(mid_data)
    X_fwd = scaler_fwd.transform(fwd_data)


    # 3. Fit PCA and transform
    X_gk_pca = pd.DataFrame(pca_gk.transform(X_gk))
    X_def_pca = pd.DataFrame(pca_def.transform(X_def))
    X_mid_pca = pd.DataFrame(pca_mid.transform(X_mid))
    X_fwd_pca = pd.DataFrame(pca_fwd.transform(X_fwd))

    X_pca = [X_gk_pca, X_def_pca, X_mid_pca, X_fwd_pca]

    print('data successfully cleaned...')

    return data, prices, xPts, teams, ids, X_pca

def generate_probs(X_pca, betas):
    print('starting to generate selction probabilities')
    def safe_alpha(X_pca, beta):
        raw_scores = pm.math.dot(X_pca, beta.T)
        raw_scores -= pm.math.max(raw_scores)
        alpha = pm.math.exp(raw_scores) + 1e-3
        alpha = pm.math.clip(alpha, 1e-3, 1e3)
        return alpha


    alpha_g = safe_alpha(X_pca[0], betas[0].T)
    alpha_d = safe_alpha(X_pca[1], betas[1].T)
    alpha_m = safe_alpha(X_pca[2], betas[2].T)
    alpha_f = safe_alpha(X_pca[3], betas[3].T)

    with pm.Model():
        pg = pm.Dirichlet("pg", a=alpha_g)  # Ensure correct shape
        pd_ = pm.Dirichlet("pd_", a=alpha_d)  # Ensure correct shape
        pmid = pm.Dirichlet("pm", a=alpha_m)  # Ensure correct shape
        pf = pm.Dirichlet("pf", a=alpha_f)  # Ensure correct shape

        trace = pm.sample()

    # Extract posterior means
    pgk, pdef, pmid, pfwd = (
        trace.posterior["pg"].mean(dim=("chain", "draw")).values,
        trace.posterior["pd_"].mean(dim=("chain", "draw")).values,
        trace.posterior["pm"].mean(dim=("chain", "draw")).values,
        trace.posterior["pf"].mean(dim=("chain", "draw")).values)

    return pgk, pdef, pmid, pfwd

def generate_W0(pg, pdef, pmid, pf, ids, prices, teams, q ,  Blb, budget=100, squads=5000):
    print('Generating Wo...')
    cin = 11  # Starting lineup size

    q = 0.25  # Assume 30% stacking probability
    g_ids = ids[0]
    d_ids = ids[1]
    m_ids = ids[2]
    f_ids = ids[3]

    g_prices = prices[0]
    d_prices = prices[1]
    m_prices = prices[2]
    f_prices = prices[3]

    g_teams = teams[0]
    d_teams = teams[1]
    m_teams = teams[2]
    f_teams = teams[3]

    gk_sizes = [1]
    def_sizes = [3,4,5]
    mid_sizes = [2,3,4,5]
    fwd_sizes = [1,2,3]

    Wo = set()
    # === Step 3: Generate Possible Team Selections (Algorithm 1) ===
    def generate_squad(pg, pdef, pmid, pf, q, budget):
        """Generate a valid team selection for an opponent."""
        stack = np.random.binomial(1, q)  # Stacking decision

        # print('===============================> Adding Goalkeeper')
        while True:
            selected_g = np.random.multinomial(1, pg)
            if set(selected_g) <= {0, 1}:  # Ensure only 1s and 0s exist
                break
        g_cost = np.dot(selected_g, g_prices)[0]
        g_team_idx = [idx for idx, _ in enumerate(selected_g) if selected_g[idx] == 1]
        selected_team_g = [g_teams[i] for i in g_team_idx] # Get the selected teams
        squad_g = [g_ids[i] for i in g_team_idx]

        for i in def_sizes:
            # print('===============================> Adding Defender', i)
            while True: # substitute for a do_while loop
                while True:# Ensure only 1s and 0s exist
                    selected_d = np.random.multinomial(i, pdef)
                    if set(selected_d) <= {0, 1}:  # Ensure only 1s and 0s exist
                        break

                d_cost = np.dot(selected_d, d_prices)[0]
                d_team_idx = [idx for idx, _ in enumerate(selected_d) if selected_d[idx] == 1]
                selected_team_g_d = selected_team_g + [d_teams[i] for i in d_team_idx] # Get the selected teams
                squad_g_d = squad_g + [d_ids[i] for i in d_team_idx]

                if ((not any(team >=3 for team in Counter(selected_team_g_d).values())) and g_cost + d_cost < Blb):
                    # Check if no more than 3 players from the same team are selected
                    # If the cost is below the lower bound, break and continue to the next iteration
                    break

            for j in mid_sizes:
                # print('===============================> Adding Midfielder ',i,  j)
                while True: # substitute for a do_while loop
                    while True:
                        selected_m = np.random.multinomial(j, pmid)
                        if set(selected_m) <= {0, 1}:  # Ensure only 1s and 0s exist
                            break

                    m_cost = np.dot(selected_m, m_prices)[0]
                    m_team_idx = [idx for idx, _ in enumerate(selected_m) if selected_m[idx] == 1]
                    selected_team_g_d_m = selected_team_g_d + [m_teams[i] for i in m_team_idx] # Get the selected teams
                    squad_g_d_m = squad_g_d + [m_ids[i] for i in m_team_idx]

                    if  ((not any(team >=3 for team in Counter(selected_team_g_d_m).values())) and g_cost + d_cost + m_cost < Blb):
                        # Check if no more than 3 players from the same team are selected
                        # If the cost is below the lower bound, break and continue to the next iteration
                        break

                for k in fwd_sizes:
                    # print('===============================> Adding Forward ',i,  j, k)
                    if(i+j+k < 10 or i+j+k > 10):
                        continue

                    while True:
                        while True:
                            selected_f = np.random.multinomial(k, pf)
                            if set(selected_f) <= {0, 1}:  # Ensure only 1s and 0s exist
                                break
                        f_cost = np.dot(selected_f, f_prices)[0]

                        f_team_idx = [idx for idx, _ in enumerate(selected_f) if selected_f[idx] == 1]
                        print(i)
                        selected_team_g_d_m_f = selected_team_g_d_m + [f_teams[i] for i in f_team_idx] # Get the selected teams

                        squad_g_d_m_f = squad_g_d_m + [f_ids[i] for i in f_team_idx]

                        squad_cost = g_cost + d_cost + m_cost + f_cost
                        print(squad_cost)
                        if((not any(team >=3 for team in Counter(selected_team_g_d_m_f).values())) and squad_cost < Blb): # 70 <

                        # if((not any(team >=3 for team in Counter(selected_team_g_d_m_f).values())) and g_cost + d_cost + m_cost + f_cost < Blb):

                            # Check if no more than 3 players from the same team are selected
                            # If the cost is below the lower bound, break and continue to the next iteration
                            Wo.add(tuple(squad_g_d_m_f))
                            break

    # squad = generate_squad(pg, pdef, pmid, pf, q, budget)
    len(Wo)
    # Generate squads possible squads
    while len(Wo)  < squads:
        generate_squad(pg, pdef, pmid, pf, q, budget)
        print(f'-------------------------------> {len(Wo)} ===> {(len(Wo)/squads)*100}%')
    # Print example squad
    print("Example Generated Squad:", len(Wo))

    print('Wo successfully created...')
    return Wo

def generate_Wo(probs, ids, prices, players_preds, budget=100, squads=5000):
    Wo = set()
    num_squads_to_generate = squads

    pbar = tqdm(total=num_squads_to_generate, desc="Generating Squads")

    # budget = 100
    formations = [(3, 4, 3), (3, 5, 2), (4, 3, 3), (4, 4, 2),
                    (4, 5, 1), (5, 2, 3), (5, 3, 2), (5, 4, 1)]

    attempts = 0
    max_attempts = num_squads_to_generate * 50 # Increase attempts if it's difficult

    while len(Wo) < num_squads_to_generate and attempts < max_attempts:
        attempts += 1

        squad_ids = []
        current_cost = 0.0
        team_counts = collections.defaultdict(int)

        num_def, num_mid, num_fwd = random.choice(formations)
        squad_template = [ 1, num_def, num_mid,  num_fwd]

        is_squad_possible = True

        for position, num_to_pick in enumerate(squad_template):
            # 1. Filter candidates based on current squad state
            candidate_ids = []
            candidate_probs = []

            # Get all players for the current position
            all_pos_ids = ids[position] #['ids']
            all_pos_probs = probs[position] #['probs']
            all_pos_prices = prices[position]['value']

            for player_id, prob, price in zip(all_pos_ids, all_pos_probs, all_pos_prices):
                # Add player if not already in squad and plausible
                if player_id not in squad_ids and price <= (budget - current_cost):
                    candidate_ids.append(player_id)
                    candidate_probs.append(prob)

            if len(candidate_ids) < num_to_pick:
                is_squad_possible = False
                break

            # 2. Renormalize probabilities of valid candidates
            total_prob = sum(candidate_probs)
            if total_prob == 0:
                is_squad_possible = False
                break

            renormalized_probs = [p / total_prob for p in candidate_probs]

            # 3. Perform multinomial draw until we get unique players
            while True:
                # n=number to pick, p=probabilities
                selection_counts = np.random.multinomial(num_to_pick, renormalized_probs)
                if np.all(selection_counts <= 1):  # Ensure uniqueness
                    break

            # Get the IDs of the selected players
            chosen_ids = [cid for cid, count in zip(candidate_ids, selection_counts) if count > 0]

            # 4. Validate the new additions and update squad state
            temp_cost = current_cost
            temp_team_counts = team_counts.copy()

            for player_id in chosen_ids:
                # print(players_preds)
                player = players_preds[players_preds['FPL_ID'] == player_id]

                temp_cost += player['value'].values
                temp_team_counts[player.team.iloc[0]] += 1

            if temp_cost> budget or any(c > 3 for c in temp_team_counts.values()):
                is_squad_possible = False
                break
            else:
                squad_ids.extend(chosen_ids)
                current_cost = temp_cost
                team_counts = temp_team_counts

            # print('done seelcting: ', position)
        if is_squad_possible and len(squad_ids) == 11:
            squad_frozenset = frozenset(squad_ids)
            if squad_frozenset not in Wo:
                Wo.add(squad_frozenset)
                pbar.update(1)
    pbar.close()

    print(f"\nSuccessfully generated {len(Wo)} unique squads")
    return Wo

def generate_cov_mat(players_preds, Wo, player_ids, data_24_25):
    print('Starting to optimize team selection...')
    xp_lookup = dict(zip(players_preds['FPL_ID'], players_preds['xP_preds']))
    G_ = [ np.sum([xp_lookup.get(id_, 0) for id_ in wo_]) for wo_ in Wo]

    Wo_ = []
    # player_ids  = players_preds['FPL_ID'].tolist()

    for i in Wo:
        # Create a tuple of 0s and 1s for each player
        squad = tuple([1 if id_ in i else 0 for id_ in player_ids])
        Wo_.append(squad)

    available_ids = players_preds['FPL_ID'].unique().tolist()
    filtered_data = data_24_25[data_24_25['element'].isin(available_ids)]

    # Group by 'event', sort the events in ascending order, and pivot the data
    pivoted_data = filtered_data.pivot_table(index='event', columns='element', values='xP')

    pivoted_data.fillna(0, inplace=True)
    pivoted_ids = pivoted_data.columns.tolist()
    extra_ids = list(set(available_ids) - set(pivoted_ids)) # Check for missing ids

    # Add extra_ids as new columns with 0 values for each event
    for element in extra_ids:
        pivoted_data[element] = 0

    pivoted_data= pivoted_data.reindex(sorted(pivoted_data.columns), axis=1)
    pivoted_data_with_extras = pivoted_data.reindex(sorted(pivoted_data.columns), axis=1)

    # pivoted_data_with_extras.shape[1]
    # num_player = pivoted_data_with_extras.shape[1]
    # cov_mat = pivoted_data_with_extras.cov()

    mu_delta = pivoted_data_with_extras.mean()
    sigma_delta = pivoted_data_with_extras.std()

    # Replace zeros with a small positive value. Ensure all standard deviations are positive
    sigma_delta = np.where(sigma_delta <= 0, 1e-6, sigma_delta)

    return Wo_ , mu_delta, sigma_delta

def monte_carlo_simulation(W_o, mu_delta, sigma_delta, num_samples=500):
    num_player = len(W_o[0])
    """Generate Monte Carlo samples for player performances and opponent scores using PyMC."""
    with pm.Model() as model:
        # Note that we access the distribution for the standard
        # deviations, and do not create a new random variable.
        sd_dist = pm.HalfNormal.dist(sigma_delta)

        mu_delta_ = pm.Deterministic('mu_delta_', pm.math.exp(mu_delta)+1e-6)
        chol, corr, sigmas = pm.LKJCholeskyCov(
            'chol_cov', eta=20, n=num_player, sd_dist=sd_dist
        )

        # Or compute the covariance matrix
        cov = pt.dot(chol, chol.T)

        # print(mu_delta_.min())
        delta = pm.MvNormal("delta", mu=mu_delta_, cov=cov, shape=num_player)

        # print(delta)

        k = int(len(W_o) * 0.5)  # Index for 50th percentile

        G_r = pm.Deterministic("G_r", pt.sort(pt.dot(W_o, delta))[-int(len(W_o) * 0.9)])  #pt.sort(pt.dot(W_o, delta), k)[k])
        # print(G_r)
        trace = pm.sample(num_samples, return_inferencedata=True, cores=2, progressbar=True)

    delta_samples = trace.posterior["delta"].values.reshape(-1, num_player)
    G_r_samples = trace.posterior["G_r"].values.flatten()

    return delta_samples, G_r_samples

def estimate_parameters(delta_samples, G_r_samples):
    """
    Estimates parameters from Monte Carlo samples.

    Parameters:
    - delta_samples: Array of shape (num_samples, num_parameters) containing samples of delta.
    - G_r_samples: Array of shape (num_samples,) containing samples of G(r').

    Returns:
    - mu_delta: Mean of delta samples.
    - Sigma_delta: Covariance matrix of delta samples.
    - mu_G_r: Mean of G(r') samples.
    - sigma2_G_r: Variance of G(r') samples.
    - sigma_delta: Standard deviations of delta samples.
    - G_r: G(r') samples.
    """

    # 1. Estimate mu_delta (Mean of delta)
    mu_delta = np.mean(delta_samples, axis=0)

    # 2. Estimate Sigma_delta (Covariance matrix of delta)
    Sigma_delta = np.cov(delta_samples, rowvar=False)

    # 3. Estimate mu_G_r (Mean of G(r'))
    mu_G_r = np.mean(G_r_samples)

    # 4. Estimate sigma2_G_r (Variance of G(r'))
    sigma2_G_r = np.var(G_r_samples, ddof=1)  # Using unbiased estimator (Bessel's correction)

    # 5. Estimate sigma_delta (Standard deviations of delta)
    sigma_delta = np.std(delta_samples, axis=0, ddof=1)  # Using unbiased estimator

    # 6. G(r') samples
    G_r = G_r_samples

    return {
        'mu_delta': mu_delta,
        'Sigma_delta': Sigma_delta,
        'mu_G_r': mu_G_r,
        'sigma2_G_r': sigma2_G_r,
        'sigma_delta': sigma_delta,
        'G_r': G_r
    }

def generate_cov_delta(delta_samples, G_r_samples):
    # Estimate μδ (mean vector of δ)
    mu_delta = np.mean(delta_samples, axis=0)  # Shape: [P]

    # Estimate Σδ (covariance matrix of δ)
    cov_delta = np.cov(delta_samples, rowvar=False)  # Shape: [P, P]

    # Estimate μG(r') (mean of G^{(r')})
    mu_G = np.mean(G_r_samples)

    # Estimate σ²G(r') (variance of G^{(r')})
    var_G = np.var(G_r_samples, ddof=1)

    sigma_G = np.std(G_r_samples)

    # Estimate σδ,G(r') (covariance vector between δ and G^{(r')})
    cov_delta_G = np.array([
        np.cov(delta_samples[:, i], G_r_samples, ddof=1)[0, 1]
        for i in range(delta_samples.shape[1])
    ])  # Shape: [P]

    return mu_delta, cov_delta, mu_G, var_G, cov_delta_G, sigma_G

def optimize(mu_delta, mu_G, var_G, sigma_G, cov_delta, gw, player_ids, players_preds, ids):
    g_ids = ids[0]
    d_ids = ids[1]
    m_ids = ids[2]
    f_ids = ids[3]

    gk_indexes = [player_ids.index(gk_id) for gk_id in g_ids if gk_id in player_ids]
    d_indexes = [player_ids.index(d_id) for d_id in d_ids if d_id in player_ids]
    m_indexes = [player_ids.index(m_id) for m_id in m_ids if m_id in player_ids]
    f_indexes = [player_ids.index(f_id) for f_id in f_ids if f_id in player_ids]

    m = Model("Portfolio_Optimization")

    # Define the number of assets
    n_assets = len(mu_delta)

    # Define the portfolio weight variables
    w = m.addMVar(shape=n_assets, vtype=GRB.BINARY, name="w")

    # Add constraints (e.g., budget constraint: sum of weights = 1)
    # Total players constraint
    m.addConstr(sum(w[idx] for idx , _ in enumerate(player_ids)) == 11, "total_players")

    # Add position limits if necessary
    # GK: exactly 1
    m.addConstr(sum(w[i] for i in gk_indexes) == 1, "gk_exactly_1")

    # DEF: between 3 and 5
    m.addConstr(sum(w[i] for i in d_indexes) >= 3, "def_min_3")
    m.addConstr(sum(w[i] for i in d_indexes) <= 5, "def_max_5")

    # MID: between 2 and 5
    m.addConstr(sum(w[i] for i in m_indexes) >= 4, "mid_min_2")
    m.addConstr(sum(w[i] for i in m_indexes) <= 5, "mid_max_5")

    # FWD: between 1 and 3
    m.addConstr(sum(w[i] for i in f_indexes) >= 3, "fwd_min_1")
    m.addConstr(sum(w[i] for i in f_indexes) <= 3, "fwd_max_3")


    # Add team constraints max 3 players from the same team
    team_ids = players_preds['player_team'].tolist()

    MAX_PER_TEAM = 3
    unique_teams = set(team_ids)

    for team in unique_teams:
        team_indexes = [i for i, t in enumerate(team_ids) if t == team]
        m.addConstr(w[team_indexes].sum() <= MAX_PER_TEAM, name=f"team_limit_{team}")

    # Assuming x is a dictionary of binary variables: x[i] for player i

    muY = sum(mu_delta[i] * w[i] for i in range(n_assets)) #- mu_G

    sigmaY_sq = sum(cov_delta[i][j] * w[i] * w[j] for i in range(n_assets) for j in range(n_assets))
    # # Variance of the score DIFFERENCE
    # team_variance = sum(cov_delta[i][j] * w[i] * w[j] for i in range(n_assets) for j in range(n_assets))
    # team_opponent_covariance = sum(cov_delta[i] * w[i] for i in range(n_assets))

    # sigmaY_sq = team_variance + var_G - (2 * team_opponent_covariance)
    # print(mu_G, var_G, '----', sigmaY_sq )

    # Set the objective function
    m.setObjective(muY - 0.5 * sigmaY_sq, GRB.MAXIMIZE)

    # Print variables and constraints before solving
    print("\n🔍 Variable Info:")
    for i in range(n_assets):
        print(f"w[{i}] ({player_ids[i]}): Binary")

    print("\n📏 Constraints:")
    for c in m.getConstrs():
        print(f"{c.ConstrName}")

    # Solve the model
    m.optimize()

    # Output solution
    if m.status == GRB.OPTIMAL:
        print("\n✅ Selected Players:")
        selected = [i for i in range(n_assets) if w[i].X > 0.8]
        for i in selected:
            print(f"- {player_ids[i]} (index {i}, expected points = {mu_delta[i]})")
    else:
        print("\n❌ No feasible solution found.")

    return selected

def selected_info(selected, players_preds, ids):
    g_ids = ids[0]
    d_ids = ids[1]
    m_ids = ids[2]
    f_ids = ids[3]

    player_ids  = players_preds['FPL_ID'].tolist()
    gk_indexes = [player_ids.index(gk_id) for gk_id in g_ids if gk_id in player_ids]
    d_indexes = [player_ids.index(d_id) for d_id in d_ids if d_id in player_ids]
    m_indexes = [player_ids.index(m_id) for m_id in m_ids if m_id in player_ids]
    f_indexes = [player_ids.index(f_id) for f_id in f_ids if f_id in player_ids]

    players_ids_details =  pd.read_csv('../FPL predictors/with new features/fpl_understat_id_name.csv')
    players_ids_details = players_ids_details.drop_duplicates(subset='fpl_id', keep='first')

    players_picked = [player_ids[i] for i in selected]
    players_picked_details = players_ids_details[players_ids_details['fpl_id'].isin(players_picked)]

    # total points
    total_points = 0
    for i in selected:
        total_points += players_preds[players_preds['FPL_ID'] == player_ids[i]]['total_points'].values[0]

    # playerr names picked
    player_names = players_picked_details['fpl_name'].tolist()

    return  players_picked, player_names, total_points

In [110]:
def get_optimized_team(gw):
    # gw=38
    data_24_25 = pd.read_csv('../FPL predictors/with new features/data/joint/24-25/merged_player_data.csv')
    players_preds = pd.read_csv(f'./../FPL predictors/with new features/models/preds/players_preds_{gw}.csv', index_col=0)
    fpl_teams_ids = pd.read_csv('../FPL predictors/with new features/data/fpl_names_24_25.csv')

    # # Add fpl_id to player_preds
    players_preds = players_preds.merge(fpl_teams_ids, left_on='fpl_name', right_on='FPL_Name', how='left')
    players_preds.dropna(subset=['FPL_ID'], inplace=True)

    player_ids  = players_preds['FPL_ID'].tolist()

    data, prices, xPts, teams, ids, X_pca = clean_data(players_preds)
    probs = generate_probs(X_pca, betas=[beta_gk, beta_def, beta_mid, beta_fwd])
    Wo = generate_Wo(probs, ids, prices, players_preds,  budget = 100, squads=1_000)
    Wo_, mu_delta, sigma_delta = generate_cov_mat(players_preds, Wo, player_ids, data_24_25)
    delta_samples, G_r_samples = monte_carlo_simulation(Wo_, mu_delta, sigma_delta, num_samples=500)
    parameters = estimate_parameters(delta_samples, G_r_samples)
    mu_delta, cov_delta, mu_G, var_G, cov_delta_G, sigma_G = generate_cov_delta(delta_samples, G_r_samples)
    selected = optimize(mu_delta, mu_G, var_G, sigma_G, cov_delta, gw, player_ids, players_preds, ids)
    players_picked, player_names, total_points = selected_info(selected, players_preds, ids)

    return players_picked, player_names, total_points

# Prev


## GW 36


In [5]:
players_picked, player_names, total_points = get_optimized_team(36)
print(players_picked, player_names, total_points)

data successfully cleaned...
starting to generate selction probabilities


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [pg, pd_, pm, pf]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 93 seconds.
There were 2000 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Generating Squads: 100%|██████████| 1000/1000 [00:08<00:00, 122.47it/s]



Successfully generated 1000 unique squads
Starting to optimize team selection...


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [chol_cov, delta]


Output()

Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 83 seconds.
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\arviz\stats\diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 494 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Set parameter Username
Set parameter LicenseID to value 2646408
Academic license - for non-commercial use only - expires 2026-04-02

🔍 Variable Info:
w[0] (517.0): Binary
w[1] (14.0): Binary
w[2] (388.0): Binary
w[3] (217.0): Binary
w[4] (81.0): Binary
w[5] (147.0): Binary
w[6] (239.0): Binary
w[7] (372.0): Binary
w[8] (247.0): Binary
w[9] (77.0): Binary
w[10] (401.0): Binary
w[11] (329.0): Binary
w[12] (310.0): Binary
w[13] (513.0): Binary
w[14] (233.0): Binary
w[15] (240.0): Binary
w[16] (335.0): Binary
w[17] (432.0): Binary
w[18] (398.0): Binary
w[19] (78.0): Binary
w[20] (365.0): Binary
w[21] (489.0): Binary
w[22] (238.0): Binary
w[23] (282.0): Binary
w[24] (146.0): Binary
w[25] (160.0): Binary
w[26] (484.0): Binary
w[27] (275.0): Binary
w[28] (24.0): Binary
w[29] (248.0): Binary
w[30] (635.0): Binary
w[31] (42.0): Binary
w[32] (285.0): Binary
w[33] (595.0): Binary
w[34] (366.0): Binary
w[35] (394.0): Binary
w[36] (99.0): Binary
w[37] (17.0): Binary
w[38] (301.0): Binary
w[39] (421

## GW 37


In [6]:
players_picked, player_names, total_points = get_optimized_team(37)
print(players_picked, player_names, total_points)

data successfully cleaned...
starting to generate selction probabilities


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [pg, pd_, pm, pf]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 100 seconds.
There were 2000 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Generating Squads: 100%|██████████| 1000/1000 [00:08<00:00, 121.35it/s]



Successfully generated 1000 unique squads
Starting to optimize team selection...


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [chol_cov, delta]


Output()

Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 58 seconds.
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\arviz\stats\diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 683 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



🔍 Variable Info:
w[0] (517.0): Binary
w[1] (14.0): Binary
w[2] (388.0): Binary
w[3] (217.0): Binary
w[4] (81.0): Binary
w[5] (147.0): Binary
w[6] (239.0): Binary
w[7] (372.0): Binary
w[8] (247.0): Binary
w[9] (310.0): Binary
w[10] (513.0): Binary
w[11] (233.0): Binary
w[12] (383.0): Binary
w[13] (432.0): Binary
w[14] (398.0): Binary
w[15] (78.0): Binary
w[16] (255.0): Binary
w[17] (489.0): Binary
w[18] (238.0): Binary
w[19] (282.0): Binary
w[20] (146.0): Binary
w[21] (160.0): Binary
w[22] (484.0): Binary
w[23] (24.0): Binary
w[24] (248.0): Binary
w[25] (635.0): Binary
w[26] (42.0): Binary
w[27] (285.0): Binary
w[28] (595.0): Binary
w[29] (491.0): Binary
w[30] (366.0): Binary
w[31] (394.0): Binary
w[32] (99.0): Binary
w[33] (17.0): Binary
w[34] (434.0): Binary
w[35] (421.0): Binary
w[36] (241.0): Binary
w[37] (27.0): Binary
w[38] (450.0): Binary
w[39] (115.0): Binary
w[40] (368.0): Binary
w[41] (654.0): Binary
w[42] (194.0): Binary
w[43] (447.0): Binary
w[44] (101.0): Binary
w[45] (321

## GW 38


In [7]:
players_picked, player_names, total_points = get_optimized_team(38)
print(players_picked, player_names, total_points)

data successfully cleaned...
starting to generate selction probabilities


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [pg, pd_, pm, pf]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 98 seconds.
There were 2000 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Generating Squads: 100%|██████████| 1000/1000 [00:08<00:00, 116.82it/s]



Successfully generated 1000 unique squads
Starting to optimize team selection...


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [chol_cov, delta]


Output()

Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 54 seconds.
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\arviz\stats\diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 515 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



🔍 Variable Info:
w[0] (517.0): Binary
w[1] (14.0): Binary
w[2] (388.0): Binary
w[3] (217.0): Binary
w[4] (81.0): Binary
w[5] (147.0): Binary
w[6] (239.0): Binary
w[7] (247.0): Binary
w[8] (77.0): Binary
w[9] (401.0): Binary
w[10] (310.0): Binary
w[11] (233.0): Binary
w[12] (240.0): Binary
w[13] (335.0): Binary
w[14] (432.0): Binary
w[15] (398.0): Binary
w[16] (78.0): Binary
w[17] (255.0): Binary
w[18] (489.0): Binary
w[19] (238.0): Binary
w[20] (282.0): Binary
w[21] (146.0): Binary
w[22] (484.0): Binary
w[23] (687.0): Binary
w[24] (24.0): Binary
w[25] (248.0): Binary
w[26] (42.0): Binary
w[27] (285.0): Binary
w[28] (595.0): Binary
w[29] (491.0): Binary
w[30] (366.0): Binary
w[31] (394.0): Binary
w[32] (99.0): Binary
w[33] (17.0): Binary
w[34] (434.0): Binary
w[35] (421.0): Binary
w[36] (27.0): Binary
w[37] (450.0): Binary
w[38] (115.0): Binary
w[39] (368.0): Binary
w[40] (475.0): Binary
w[41] (194.0): Binary
w[42] (447.0): Binary
w[43] (370.0): Binary
w[44] (101.0): Binary
w[45] (283.

# Current


In [111]:
def get_optimized_team(sns, gw):
    # gw=38
    data_25_26 = pd.read_csv(f'../FPL predictors/with new features/data/joint/{sns}/merged_player_data.csv')
    players_preds = pd.read_csv(f'../FPL predictors/with new features/models/preds/players_preds_{gw}.csv', index_col=0)
    sns_split = '_'.join(sns.split('-'))
    fpl_teams_ids = pd.read_csv(f'../FPL predictors/with new features/data/vaastav/data/id_dict_{sns_split}.csv')

    # print(players_preds['fpl_name'])
    # # Add fpl_id to player_preds
    players_preds = players_preds.merge(fpl_teams_ids, left_on='fpl_name', right_on='FPL_Name', how='left')
    players_preds.dropna(subset=['FPL_ID'], inplace=True)

    player_ids  = players_preds['FPL_ID'].tolist()

    data, prices, xPts, teams, ids, X_pca = clean_data(players_preds)
    probs = generate_probs(X_pca, betas=[beta_gk, beta_def, beta_mid, beta_fwd])
    Wo = generate_Wo(probs, ids, prices, players_preds,  budget = 100, squads=1_000)
    Wo_, mu_delta, sigma_delta = generate_cov_mat(players_preds, Wo, player_ids, data_25_26)
    delta_samples, G_r_samples = monte_carlo_simulation(Wo_, mu_delta, sigma_delta, num_samples=500)
    parameters = estimate_parameters(delta_samples, G_r_samples)
    mu_delta, cov_delta, mu_G, var_G, cov_delta_G, sigma_G = generate_cov_delta(delta_samples, G_r_samples)
    selected = optimize(mu_delta, mu_G, var_G, sigma_G, cov_delta, gw, player_ids, players_preds, ids)
    players_picked, player_names, total_points = selected_info(selected, players_preds, ids)

    return players_picked, player_names, total_points

In [119]:
players_picked, player_names, total_points = get_optimized_team('25-26',6)
print(players_picked, player_names, total_points)

c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\sklearn\utils\_array_api.py:472: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\sklearn\utils\_array_api.py:489: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\sklearn\utils\_array_api.py:472: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\sklearn\utils\_array_api.py:489: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\sklearn\utils\_array_api.py:472: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\sklearn\utils\_array_api.py:489: RuntimeWarning: 

data successfully cleaned...
starting to generate selction probabilities


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [pg, pd_, pm, pf]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 194 seconds.
There were 2000 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Generating Squads: 100%|██████████| 1000/1000 [00:11<00:00, 84.41it/s]



Successfully generated 1000 unique squads
Starting to optimize team selection...


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [chol_cov, delta]


Output()

Sampling 2 chains for 1_000 tune and 500 draw iterations (2_000 + 1_000 draws total) took 262 seconds.
c:\Users\Ilyas\anaconda3\envs\Pycaret\Lib\site-packages\arviz\stats\diagnostics.py:592: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 503 divergences after tuning. Increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



🔍 Variable Info:
w[0] (366): Binary
w[1] (431): Binary
w[2] (139): Binary
w[3] (314): Binary
w[4] (101): Binary
w[5] (1): Binary
w[6] (253): Binary
w[7] (32): Binary
w[8] (565): Binary
w[9] (182): Binary
w[10] (287): Binary
w[11] (628): Binary
w[12] (341): Binary
w[13] (679): Binary
w[14] (33): Binary
w[15] (470): Binary
w[16] (502): Binary
w[17] (469): Binary
w[18] (220): Binary
w[19] (670): Binary
w[20] (627): Binary
w[21] (67): Binary
w[22] (116): Binary
w[23] (610): Binary
w[24] (406): Binary
w[25] (73): Binary
w[26] (74): Binary
w[27] (372): Binary
w[28] (316): Binary
w[29] (447): Binary
w[30] (685): Binary
w[31] (11): Binary
w[32] (318): Binary
w[33] (261): Binary
w[34] (375): Binary
w[35] (662): Binary
w[36] (569): Binary
w[37] (476): Binary
w[38] (256): Binary
w[39] (682): Binary
w[40] (574): Binary
w[41] (144): Binary
w[42] (440): Binary
w[43] (573): Binary
w[44] (603): Binary
w[45] (631): Binary
w[46] (111): Binary
w[47] (474): Binary
w[48] (511): Binary
w[49] (148): Binary


In [ ]:
import requests
recoms = [
    'Noni Madueke', 'Eberechi Eze',
    'Chris Richards', 'Nathaniel Clyne', 'Bernd Leno',
    'Emile Smith Rowe', 'Lukas Nmecha', 'Ibrahim Sangaré',
    'Niclas Füllkrug', 'Jørgen Strand Larsen', 'David Møller Wolfe']

fpl = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/').json()
fpl = pd.DataFrame(fpl['elements'])

In [5]:
fpl['total_points']

0      28
1       0
2       0
3       0
4      38
       ..
737    11
738     6
739     5
740    10
741     5
Name: total_points, Length: 742, dtype: int64

In [6]:
ids = pd.read_csv('../FPL predictors/with new features/data/vaastav/data/id_dict_25_26.csv')

total_points = 0

for player in recoms:
    fpl_id = ids[ids['FPL_Name'] == player]['FPL_ID'].values[0]
    points = fpl[fpl['id'] == fpl_id]['total_points'].values[0]
    print(player, points)
    total_points += points

print('Total points: ', total_points)

Noni Madueke 10
Eberechi Eze 16
Chris Richards 30
Nathaniel Clyne 1
Bernd Leno 18
Emile Smith Rowe 11
Lukas Nmecha 11
Ibrahim Sangaré 11
Niclas Füllkrug 8
Jørgen Strand Larsen 7
David Møller Wolfe 6
Total points:  129
